# Convergence per Attack per Algorithm

In [1]:
%load_ext autoreload
import sys
import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta
import pandas as pd
import seaborn as sns
from pathlib import Path
from fastnanoid import generate
from datetime import datetime

In [2]:
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))
lib_path = project_root / "lib"
sys.path.insert(0, str(lib_path))        
sys.path.insert(0, str(project_root))   

In [3]:
%autoreload 2
from lib.graph_factory import GraphFactory, add_graph_plot
from lib.proj_const_estimator import ProjConstEstimator
from lib.utils import fetch_dataset, get_alphas, rng, seed
from lib.classifier import ByzClassifier
from lib.system import SystemSimulator
from lib.config import BASE_CONF, NUM_NODES

In [4]:
# Helper Functions
fmt = lambda sec : f"{sec // 60}m {sec % 60:.2f}s"
def convert(o):
    if isinstance(o, np.generic):
        return o.item()
    raise TypeError(f'{type(o)} not serializable')

In [5]:
RUN_DIR = Path().resolve()
config = BASE_CONF
config['train']['b'] = 7
config['sys']['b'] = config['train']['b'] 
config['classifier_model_type'] = 'mlp'

b = config['train']['b']
config['graph_args'] = {
    'ws_k': 6,
    'ws_p': 0.1,
    'rand_reg_deg':b+1,
    'geom_radius':0.35,
    'er_p':min(1.0, 3.0 * np.log(NUM_NODES) / NUM_NODES),
}
target_pi = rng('dir','exp11').dirichlet(np.full(NUM_NODES,1))
config['graph_weights'] = 'MH'
graphs = ['random-regular','watts-strogatz', 'geometric']
ATKS = ['label_flip', 'sign_flip', 'gaussian', 'ALIE', 'IPM']
# ALGS = ['RDSGD', 'ORACLE', 'IOS', 'SCC', 'TriMean', 'CooMed']
ALGS = ['RDSGD', 'ORACLE']
COLORS = dict(zip(ALGS, ['C0', 'C1', 'C2', 'C3', 'C4', 'C5']))
clf_optimal_op_pt = {
    ('random-regular','all_atks',5) : dict(cost_fp=6.0, cost_fn=1.0), 
    ('erdos-renyi', 'all_atks',5) : dict(cost_fp=6.0, cost_fn=1.0),
    ('geometric','sign_flip',10) : dict(cost_fp=3.0, cost_fn=1.0),
    ('geometric','gaussian',10) : dict(cost_fp=1, cost_fn=2),
    ('geometric','label_flip',10): dict(cost_fp=1, cost_fn=1.2),
    ('watts-strogatz','sign_flip',5): dict(cost_fp=3, cost_fn=1),
    ('watts-strogatz','ALIE',5): dict(cost_fp=3, cost_fn=1),
    ('watts-strogatz','label_flip',5): dict(cost_fp=1, cost_fn=0.9), 
}
# for random-regular graph, for all attacks the clf_optimal_op_pt have params dict(cost_fp=6.0, cost_fn=1.0) producing optimal tau_C's

In [6]:
global_dataset = fetch_dataset('MNIST')
gf = GraphFactory(config['train']['num_nodes'], b)
proj_const_estimator = ProjConstEstimator(config, global_dataset, gf) 
clf = ByzClassifier(config, global_dataset, gf)
sys_sim = SystemSimulator(config, global_dataset, gf)

In [7]:
# Fix a graph topology and attack
config['graph_type']='random-regular'
atk = 'label_flip'
RUN_ID=19
local_seed = seed(777, config['graph_type'], atk, RUN_ID)

In [8]:
proj_const_estimator.configure(config, seed('proj',local_seed))
proj_const = proj_const_estimator.estimate()

In [9]:
%%time
clf_sim_data = clf.run_simulations(config, proj_const, seed('clf',local_seed))

Simulating RDSGD @ (beta_C=0.1, gamma_C=0.12)
Building training set. Simulating ['label_flip', 'sign_flip', 'gaussian', 'ALIE', 'IPM'] (seed 56196281461)
Building validation set I. Simulating ['label_flip', 'sign_flip', 'gaussian', 'ALIE', 'IPM'] (seed 36362299755)
Building validation set II. Simulating ['label_flip', 'sign_flip', 'gaussian', 'ALIE', 'IPM'] (seed 2039594449875)
Building test set. Simulating ['label_flip', 'sign_flip', 'gaussian', 'ALIE', 'IPM'] (seed 717329004269)
CPU times: user 38min 34s, sys: 55.9 s, total: 39min 30s
Wall time: 3min 18s


In [10]:
# %%capture
# clf_mod_fig, clf_mod_ax = plt.subplots(1,1,figsize=(6,4))

In [11]:
# %%time
# models = ['rbf', 'lin-lr', 'mlp', 'xgb']
# rows = []
# for m in models:
#     print(f'Fitting {m} model...')
#     _ = clf.fit(in_model=m)
#     clf_op_pt = clf.calc_optimal_op_pt(cost_fp=1, cost_fn=2)
#     clf_metrics = clf.test()
#     clf.plot_pr(clf_mod_ax, lab=m)
#     clf_mod_ax.axhline(clf.prevalence, color='k', linestyle='--')
#     rows.append(dict(model=m,
#                      val_tau=clf_op_pt['C_tau'],
#                      val_ap=clf_op_pt['cv_ap'],
#                      test_beta=clf_metrics['beta_C'],
#                      test_gamma=clf_metrics['gamma_C'],
#                      op_prec=clf_metrics['op_prec'],
#                      op_rec=clf_metrics['op_rec'],
#                      test_avg_prec=clf_metrics['avg_prec'], 
#                      test_roc_auc=clf_metrics['roc_auc']))

In [12]:
# df_models = pd.DataFrame(rows)
# df_models.sort_values('val_ap',inplace=True,ascending=False)
# df_models.set_index('model',inplace=True)
# df_models.to_csv(f'{config['graph_type']}_{atk}_{RUN_ID}_models.csv')
# display(df_models)

In [13]:
# %%capture
# best_model = df_models.iloc[0]
# clf_mod_ax.scatter([best_model['op_rec']], [best_model['op_prec']], color='red')

In [14]:
# clf_mod_ax.set_xlim([0.0, 1.0])
# clf_mod_ax.set_ylim([0.0, 1.05])
# clf_mod_ax.set_xlabel('Recall (1 − FNR)')
# clf_mod_ax.set_ylabel('Precision')
# clf_mod_ax.set_title('Byzantine Classifier PR Curve')
# clf_mod_ax.legend(loc='lower left')
# clf_mod_fig.savefig(f'{config['graph_type']}_{atk}_{RUN_ID}_models.png')
# display(clf_mod_fig)

In [ ]:
%%time
config['classifier_model_type'] = 'rbf' # best_model.name
clf_est, clf_preproc = clf.fit(in_model='rbf')

In [ ]:
clf_op_pt = clf.calc_optimal_op_pt(cost_fp=5, cost_fn=1)
clf_metrics = clf.test()

In [ ]:
prelim_metrics = dict()
prelim_metrics['proj_const'] = proj_const
prelim_metrics.update(clf_op_pt)
prelim_metrics.update(clf_metrics)
display(pd.DataFrame([prelim_metrics]))

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(6,4))
clf.plot_pr(ax)
ax.axhline(clf.prevalence, color='k', linestyle='--')
plt.show()

In [ ]:
%%time
sim_params = {
    'algorithms': ALGS,
    'atk_type': atk,
    'threat_model': 'T3',
    'seed': local_seed
}
params_C = dict()

# Specifies RDSGD_ORACLE Parameters
params_C['C_fpr'] = clf_metrics['gamma_C']
params_C['C_fnr'] = clf_metrics['beta_C']

# Specifies RDSGD Parameters
params_C['C_tau'] = clf_op_pt['C_tau']

sys_sim.init_simulation(config, proj_const, clf_preproc, clf_est, params_C, seed(atk, local_seed))
df = sys_sim.simulate(sim_params)

In [ ]:
df.to_csv(f'{config['graph_type']}_{atk}_results_dataset_{RUN_ID}.csv')

In [ ]:
# Plot Lyapunov Function
fig, ax = plt.subplots(1,1,figsize=(6,4))
for alg in ALGS:
    if alg == 'RDSGD':
        gap = df.loc['RDSGD','opt_gap_pi']
        conv = df.loc['RDSGD','C_pi']
    else:
        gap = df.loc[alg,'opt_gap']
        conv = df.loc[alg,'C_unif']
    ax.plot(conv+gap, label=alg)
ax.legend()
ax.set(title=f"Lyapunov Convergence ({atk} attack)", xlabel='Iteration (k)', ylabel='Lyapunov Function (V)')
plt.yscale('log')
plt.show()

In [ ]:
# Plot Realized FPR and FNR 
fig, ax = plt.subplots(1,2,figsize=(10,4))
for idx, alg in enumerate(['RDSGD', 'ORACLE']):
    ax[0].plot(df.loc[alg, 'gamma_k'].ewm(com=2, adjust=True).mean(), label=f'{alg}_FPR')
    ax[0].plot(df.loc[alg, 'beta_k'].ewm(com=2, adjust=True).mean(), label=f'{alg}_FNR')
    ax[1].plot(df.loc[alg, 'byz_w_k'].ewm(com=2, adjust=True).mean(), label=alg)
ax[0].set(title=f'Realized Operating Point ({atk})', xlabel='Iteration (k)', ylabel='')
ax[1].set(title=f'Admitted Byzantine Mass ({atk})', xlabel='Iteration (k)', ylabel='')
ax[0].legend()
ax[1].legend()
plt.show()
fig.savefig(f'{config['graph_type']}_{atk}_{RUN_ID}_realized_fpr_fnr.png')

In [ ]:
df2 = pd.DataFrame()
# min across nodes of test accuracy at final iteration
df2['lb_node_ta'] = df['min_test_acc'].groupby(['alg']).nth(-1).groupby(['alg']).first() 

# max across nodes of test accuracy at final iteration 
df2['ub_node_ta'] = df['max_test_acc'].groupby(['alg']).nth(-1).groupby(['alg']).first() 

# mean test accuracy taken over system
ser = df['test_acc'].groupby(['alg']).agg('mean')
ser['RDSGD'] = df.loc['RDSGD','test_acc_pi'].mean()
ser['ORACLE'] = df.loc['ORACLE','test_acc_pi'].mean()
df2['sys_mta'] = ser     

In [ ]:
df2.sort_values('sys_mta', inplace=True, ascending=True)
display(df2)
df2.to_csv(f'{config['graph_type']}_{atk}_{RUN_ID}_lb-ub-ta_data.csv')

In [ ]:
df2 = df2.sort_values('lb_node_ta')

fig, ax = plt.subplots(figsize=(8, 6))

y_pos = np.arange(len(df2))
bar_widths = df2['ub_node_ta'] - df2['lb_node_ta']
ax.barh(y_pos, width=bar_widths, left=df2['lb_node_ta'],height=0.5, 
        color='skyblue', edgecolor='navy', alpha=0.7, label='LB-UB Range')

ax.scatter(df2['sys_mta'], y_pos, color='red', marker='D', s=80,label='asymptotic TA', zorder=5)

ref_line = df2['sys_mta'].mean()
ax.axvline(x=ref_line, color='gray', linestyle='--', linewidth=1.5, label=f'Mean TA = {ref_line:.3f}')

ax.set_yticks(y_pos)
ax.set_yticklabels(df2.index)
ax.set_xlabel('Test Accuracy')
ax.set_xlabel('Value')
ax.set_title(f'Inter-node TA Range and Asymptotic TA (T3:{atk})')
ax.legend()
ax.grid(axis='x', alpha=0.3)
fig.savefig(f'{config['graph_type']}_{atk}_{RUN_ID}_inter_node_TA.png')
plt.tight_layout()
plt.show()

In [ ]:
config.update(prelim_metrics)
with open(f'{config['graph_type']}_{atk}_{RUN_ID}_config.json', 'w') as f:
    json.dump(config, f, indent=2, default=convert)